# Frozen candidate validation-only CUDA retraining

Run this notebook from top to bottom in a fresh Colab GPU runtime. Repository synchronization and the complete 50-run preflight happen before the interactive execution lock. Existing anchor runs remain read-only.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

CONTENT_ROOT = Path('/content')
REPO_URL = 'https://github.com/khyitty/VitalDB-Feature-Selection.git'
REPO_DIR = CONTENT_ROOT / 'VitalDB-Feature-Selection'
REQUIRED_WORKFLOW_COMMIT = '8df2802'
ROOT = Path('/content/drive/MyDrive/VitalDB-Feature-Selection')
DATASET_DIR = ROOT / 'data/modeling/full'
GROUP_ROOT = ROOT / 'outputs/group_retraining_validation_only'
GROUP_ANALYSIS_DIR = GROUP_ROOT / 'analysis'
SELECTION_DIR = ROOT / 'outputs/predictive_feature_selection_30s'
CANDIDATE_PATH = SELECTION_DIR / 'candidate_subsets.json'
OUTPUT_ROOT = ROOT / 'outputs/frozen_candidate_retraining_validation_only'
RUN_FULL_TRAINING = False
CONFIRMATION_TEXT = ''
REQUIRED_CONFIRMATION = 'RUN_20_FROZEN_CANDIDATE_CUDA_RUNS'
PREFLIGHT_PASSED = False

In [ ]:
import importlib
import os
import shlex
import shutil
import subprocess
import sys

def run_command(command, *, cwd=None, check=True, stream=False):
    command = [str(part) for part in command]
    print('$', shlex.join(command))
    result = subprocess.run(
        command, cwd=cwd, check=False, capture_output=not stream, text=True
    )
    if result.stdout:
        print('[stdout]')
        print(result.stdout.rstrip())
    if result.stderr:
        print('[stderr]')
        print(result.stderr.rstrip())
    if check and result.returncode != 0:
        raise RuntimeError(
            f'Command failed with exit status {result.returncode}: {shlex.join(command)}\n'
            f'stdout:\n{result.stdout or "<empty>"}\n'
            f'stderr:\n{result.stderr or "<empty>"}'
        )
    return result

def verify_repository_state(repo_dir, required_ancestor):
    local_head = run_command(['git', '-C', repo_dir, 'rev-parse', 'HEAD']).stdout.strip()
    origin_head = run_command(['git', '-C', repo_dir, 'rev-parse', 'origin/main']).stdout.strip()
    remote_url = run_command(['git', '-C', repo_dir, 'remote', 'get-url', 'origin']).stdout.strip()
    branch_result = run_command(['git', '-C', repo_dir, 'branch', '--show-current'], check=False)
    branch = branch_result.stdout.strip() or '<detached>'
    if local_head != origin_head:
        raise RuntimeError(f'Repository is not synchronized: HEAD={local_head}, origin/main={origin_head}.')
    ancestor = run_command(
        ['git', '-C', repo_dir, 'merge-base', '--is-ancestor', required_ancestor, 'HEAD'],
        check=False,
    )
    if ancestor.returncode != 0:
        raise RuntimeError(
            f'Required workflow commit {required_ancestor} is not an ancestor of HEAD {local_head}. '
            f'Git stderr: {ancestor.stderr or "<empty>"}'
        )
    recent = run_command(['git', '-C', repo_dir, 'log', '-3', '--oneline']).stdout.strip()
    print('Repository state:', {'HEAD': local_head, 'origin/main': origin_head, 'origin': remote_url, 'branch': branch})
    print('Recent commits:\n' + recent)
    return {'head': local_head, 'origin_main': origin_head, 'origin_url': remote_url, 'branch': branch}

def synchronize_repository(repo_dir, repo_url, required_ancestor, *, allowed_parent):
    repo_dir = Path(repo_dir)
    allowed_parent = Path(allowed_parent).resolve()
    if repo_dir.parent.resolve() != allowed_parent:
        raise RuntimeError(f'Refusing repository operation outside {allowed_parent}: {repo_dir}')
    allowed_parent.mkdir(parents=True, exist_ok=True)
    os.chdir(allowed_parent)
    git_dir = repo_dir / '.git'
    if git_dir.is_dir():
        current_remote = run_command(
            ['git', '-C', repo_dir, 'remote', 'get-url', 'origin'], check=False
        )
        if current_remote.returncode != 0:
            run_command(['git', '-C', repo_dir, 'remote', 'add', 'origin', repo_url])
        elif current_remote.stdout.strip() != repo_url:
            print(f'Updating origin URL: {current_remote.stdout.strip()} -> {repo_url}')
            run_command(['git', '-C', repo_dir, 'remote', 'set-url', 'origin', repo_url])
        run_command(['git', '-C', repo_dir, 'fetch', 'origin', 'main', '--prune'])
        run_command(['git', '-C', repo_dir, 'reset', '--hard', 'origin/main'])
    else:
        if repo_dir.exists() or repo_dir.is_symlink():
            print('Replacing non-Git temporary repository path:', repo_dir)
            if repo_dir.is_symlink():
                repo_dir.unlink()
            else:
                shutil.rmtree(repo_dir)
        run_command(['git', 'clone', repo_url, repo_dir])
    return verify_repository_state(repo_dir, required_ancestor)

def prepare_project_imports(repo_dir, prefixes=('src', 'scripts')):
    stale = [
        name for name in list(sys.modules)
        if any(name == prefix or name.startswith(prefix + '.') for prefix in prefixes)
    ]
    if stale:
        print('Purging stale project modules before re-import:', sorted(stale))
        for name in stale:
            sys.modules.pop(name, None)
    resolved_repo = Path(repo_dir).resolve()
    retained = []
    for entry in sys.path:
        try:
            if Path(entry or os.curdir).resolve() == resolved_repo:
                continue
        except OSError:
            pass
        retained.append(entry)
    sys.path[:] = [str(resolved_repo), *retained]
    importlib.invalidate_caches()
    return stale

REPOSITORY_STATE = synchronize_repository(
    REPO_DIR, REPO_URL, REQUIRED_WORKFLOW_COMMIT, allowed_parent=CONTENT_ROOT
)
PURGED_PROJECT_MODULES = prepare_project_imports(REPO_DIR)
os.chdir(REPO_DIR)

In [ ]:
run_command(
    [sys.executable, 'scripts/install_colab_dependencies.py', '--requirements', 'requirements-colab.txt'],
    cwd=REPO_DIR,
)

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU, then use Run all again.')
ACTIVE_COMMIT = REPOSITORY_STATE['head']
print('Commit:', ACTIVE_COMMIT)
print('GPU:', torch.cuda.get_device_name(0))

## Validate source code, frozen candidates, and 30 reused anchors

This preflight is read-only for prior group runs and stops before the training confirmation prompt on any incompatibility.

In [ ]:
import importlib
import json

state = verify_repository_state(REPO_DIR, REQUIRED_WORKFLOW_COMMIT)
if state['head'] != ACTIVE_COMMIT:
    raise RuntimeError(f'Active code changed after bootstrap: {ACTIVE_COMMIT} -> {state["head"]}')
workflow = importlib.import_module('src.frozen_candidate_retraining')
module_path = Path(workflow.__file__).resolve()
if not module_path.is_relative_to(REPO_DIR.resolve()):
    raise RuntimeError(f'Stale frozen-candidate module loaded from {module_path}. Restart runtime.')
for function_name in ('resolve_git_commit', 'require_same_git_commit'):
    if not hasattr(workflow, function_name):
        raise RuntimeError(f'Active source lacks required SHA resolver: {function_name}')

prior_config_path = GROUP_ROOT / 'full17/gru/seed_7/config.json'
prior_config = json.loads(prior_config_path.read_text(encoding='utf-8'))
for field in ('git_commit_hash', 'seed', 'device', 'evaluate_test', 'dynamic_feature_names'):
    print(f'prior {field}:', prior_config.get(field))
expected_full = workflow.resolve_git_commit(
    REPO_DIR, workflow.TRAINING_COMMIT, label='expected group-training commit'
)
observed_full = workflow.resolve_git_commit(
    REPO_DIR, prior_config.get('git_commit_hash'), label='observed full17/gru/seed_7 commit'
)
if observed_full != expected_full:
    raise RuntimeError(
        f'Prior SHA mismatch: expected {workflow.TRAINING_COMMIT} -> {expected_full}; '
        f'observed {prior_config.get("git_commit_hash")} -> {observed_full}'
    )
print('Prior short/full SHA compatibility:', expected_full)

anchor_directories = [
    GROUP_ROOT / 'full17',
    GROUP_ROOT / 'no_respiratory',
    GROUP_ROOT / 'no_remifentanil_or_respiratory',
]
def artifact_snapshot(directories):
    return {
        str(path.relative_to(GROUP_ROOT)): (path.stat().st_size, path.stat().st_mtime_ns)
        for directory in directories
        for path in directory.rglob('*')
        if path.is_file()
    }

anchor_before = artifact_snapshot(anchor_directories)
preflight_command = [
    sys.executable, 'scripts/run_frozen_candidate_retraining.py',
    '--candidate-subsets', CANDIDATE_PATH,
    '--dataset-dir', DATASET_DIR,
    '--group-root', GROUP_ROOT,
    '--group-analysis-dir', GROUP_ANALYSIS_DIR,
    '--output-root', OUTPUT_ROOT,
    '--repo-dir', REPO_DIR,
]
run_command(preflight_command, cwd=REPO_DIR)
anchor_after = artifact_snapshot(anchor_directories)
if anchor_before != anchor_after:
    raise RuntimeError('Preflight modified one or more existing anchor artifacts.')

PLAN = json.loads((OUTPUT_ROOT / 'experiment_plan.json').read_text(encoding='utf-8'))
REGISTRY = json.loads((OUTPUT_ROOT / 'candidate_source_registry.json').read_text(encoding='utf-8'))
reused = [row for row in REGISTRY if row['source_type'] == 'reused_prior']
new = [row for row in REGISTRY if row['source_type'] == 'newly_trained']
if (len(reused), len(new), len(REGISTRY)) != (30, 20, 50):
    raise RuntimeError(f'Unexpected inventory: reused={len(reused)}, new={len(new)}, total={len(REGISTRY)}')
if any(row.get('test_evaluated') is not False for row in REGISTRY):
    raise RuntimeError('Registry contains a run that is not test-sealed.')
for row in reused:
    run_dir = Path(row['source_run_directory'])
    forbidden = [name for name in workflow.FORBIDDEN_TEST_ARTIFACTS if (run_dir / name).exists()]
    if forbidden:
        raise RuntimeError(f'Forbidden test artifacts in {run_dir}: {forbidden}')
PREFLIGHT_PASSED = True
print('Preflight passed: reused=30, newly planned=20, registry=50, test split sealed.')
for name, features in PLAN['frozen_candidates'].items():
    print(name, len(features), features)

## Interactive execution lock

Run all pauses here. Enter the exact confirmation only after reviewing the successful preflight; no notebook cell editing is required.

In [ ]:
if not PREFLIGHT_PASSED:
    raise RuntimeError('Training cannot be unlocked before a successful preflight.')
CONFIRMATION_TEXT = input(
    f'Type {REQUIRED_CONFIRMATION} to start exactly 20 CUDA validation-only runs; '
    'press Enter to keep training locked: '
).strip()
RUN_FULL_TRAINING = CONFIRMATION_TEXT == REQUIRED_CONFIRMATION
if not RUN_FULL_TRAINING:
    raise RuntimeError('Training remains locked. No new run was started.')
print('Training unlocked: exactly 20 new CUDA validation-only runs.')

In [ ]:
from datetime import datetime, timezone
from src.colab_workflow import inspect_run_completion
from src.frozen_candidate_retraining import (
    build_training_command, dump_json, update_registry_after_run,
    validate_resume_compatibility,
)

execution = []
new_records = [row for row in REGISTRY if row['source_type'] == 'newly_trained']
if len(new_records) != 20:
    raise RuntimeError(f'Execution inventory changed after preflight: {len(new_records)} new runs.')
for record in new_records:
    run_dir = Path(record['source_run_directory'])
    run_dir.mkdir(parents=True, exist_ok=True)
    completion = inspect_run_completion(run_dir, record['model'])
    if completion['complete']:
        update_registry_after_run(REGISTRY, record, DATASET_DIR)
        action = 'skipped_complete'
        return_code = 0
    else:
        resume = validate_resume_compatibility(run_dir, record, DATASET_DIR, ACTIVE_COMMIT)
        command = build_training_command(record, DATASET_DIR, resume)
        if command[command.index('--device') + 1] != 'cuda' or '--validation-only' not in command:
            raise RuntimeError(f'Unsafe training command refused: {command}')
        result = run_command(command, cwd=REPO_DIR, check=False, stream=True)
        return_code = result.returncode
        action = 'resumed' if resume else 'started'
        if return_code != 0:
            raise RuntimeError(f'Run failed after full diagnostics above: {run_dir}')
        update_registry_after_run(REGISTRY, record, DATASET_DIR)
    execution.append({
        'candidate': record['candidate'], 'model': record['model'], 'seed': record['seed'],
        'action': action, 'return_code': return_code,
        'finished_at_utc': datetime.now(timezone.utc).isoformat(),
    })
    dump_json(REGISTRY, OUTPUT_ROOT / 'candidate_source_registry.json')
    dump_json(execution, OUTPUT_ROOT / 'execution_manifest.json')
print('Completed or verified', len(execution), 'of 20 new runs.')